**`04_filepath_tutorial`**

A tutorial for using the `openplaces` file path referencing (directory structure, prefixes)

In [ ]:
# Import path functions
from openplaces.path import external_path, models_path, path

# Usage of `path`

## Basics
- ``path`` returns a ``pathlib.Path``, the cross-platform standard in Python.
    - Actually, it returns an ``openplaces.path.OpenPlacesPath``, but that's just a ``pathlib.Path`` with a more concise display.
- By default, it returns paths to files, not directories.
    - Missing filename extension? Default is ``.parquet``.
    - Missing filename? Default is ``unnamed.parquet``.
- Directories need to be identified with ``as_dir=True``


In [ ]:
# If no arguments are provided: return a directory (no extension)

# '_all' is the escape directory for the hierarchical administrative
# directory structure. It means: here's the data for all subdivisions
# of this administrative units. At the top-level, the `_all` directory
# refers to global / planetary data.

path()

In [ ]:
# External data directory
external_path()

In [ ]:
# Shortcut: models directory
models_path()

## AdminId
The **1st** reference attribute: ``admin_id``, the identifier of the administrative unit
- Administrative unit identifiers become part of the hierarchical directory structure.
- A prefix is added to all output files to make them unique.
  - This facilitates manual data inspection and comparisons (Excel won't let you open two files by the same name).

In [ ]:
# Get the path to US data
# In absence of a `filename` argument, the prefix becomes the filename
path('US', 'transaction')

In [ ]:
# Get just the directory (skip filename creation)
path('US', as_dir=True)

In [ ]:
# Path to file in US data directory
path('US', filename='forest_map.tif')

In [ ]:
# Path to transactions from the state of Wisconsin, US
path('US-WI', filename='transactions')

In [ ]:
# Path to transactions from Dane county, Wisconsin, US
path('US-WI-DA', filename='transactions')

### Input directories
Directories with original input data (external, raw, heap) have prefixing turned off to accommodate original filenames.

In [ ]:
external_path('US', filename='downloaded_data.zip')

In [ ]:
external_path(
    admin_id='US-ND',
    dataset='water-lake-nhd-2019hr',
    filename='NHD_H_North_Dakota_State_GDB.zip',
)

## Entity
The **2st** reference attribute: ``entity``, the identifier of the basic units of analysis
(rows of a data table).
-  They can be parcels, properties, transactions, people, administrative units.

In [ ]:
# Directory of parcel data in Wisconsin
path('US-WI', 'parcel')

In [ ]:
# Directory of version 11 of Wisconsin's statewide parcel dataset
path('US-WI', 'parcel-wiscedu-v11')

## DataSet

The **3st** reference attribute: ``dataset``, the (usually spatial) dataset that the entities will get intersected with.

-  The ``theme`` is a hierarchical thematic identifier with flexible depth.
    -  First level enforces whitelist to promote shared top-level structure (e.g. ``climate``, ``water``, ``bio``, ``landcover``, ``people``, ``rules``).
-  A ``source`` provides identifiers, a link to the data portal (``portal_url``) and, where available, direct download links (``download_url``).

In [ ]:
# Path to intersected data in `openplaces` data folder
# - For United States > Massachusetts > Michigan
# - Features: water > lakes data > from the National Hydrography Dataset > High-Res
# Adds file prefixes and `.parquet` extension by default
external_path('US-MA', dataset='water-lake-nhd-2019hr')

## Intersections
References including both an entity and dataset refer to the results of geoprocessing (usually some form of spatial intersection)

In [ ]:
# Path to intersected data in `openplaces` data folder
# - In: United States > Wisconsin > Dane county
# - Entities: parcels, by MassGIS, in 2018
# - Features: water > lakes data > from the National Hydrography Dataset > High-Res
# Adds file prefixes and `.parquet` extension by default
path('US-WI-DA', 'parcel-wiscedu-v11', 'water-lake-nhd-2019hr')

In [ ]:
# Path to intersected data
# - In: United States > California
# - Entities: buildings, as per Microsoft footprints, 2019
# - Features: landcover > forest > from Hansen's dataset, 2024 update
# Adds file prefixes and `.parquet` extension by default
path(
    admin_id='US-CA',
    entity='building-microsoft-2019',
    dataset='landcover-forest-hansen-2024',
)

# Schema

## Admin ID

In [ ]:
from openplaces.path import AdminId

In [ ]:
# Global
admin_id = AdminId()
print('ID:          ', str(admin_id))
print('Path:        ', admin_id.to_path())
print('File prefix: ', admin_id.to_prefix())

In [ ]:
# Country
admin_id = AdminId('US')
print('ID:          ', str(admin_id))
print('Path:        ', admin_id.to_path())
print('File prefix: ', admin_id.to_prefix())

In [ ]:
# State, department
admin_id = AdminId('US', 'MA')
print(admin_id)
print(admin_id.to_path())
print(admin_id.to_prefix())

In [ ]:
# State, department
admin_id = AdminId('US-MA')
print(admin_id)
print(admin_id.to_path())
print(admin_id.to_prefix())

In [ ]:
# County, municipality
admin_id = AdminId('US-MA-MI')
print(admin_id)
print(admin_id.to_path())
print(admin_id.to_prefix())

In [ ]:
# Town, etc.
admin_id = AdminId('US-MA-SOM')
print(admin_id)
print(admin_id.to_path())
print(admin_id.to_prefix())

## Entity

In [ ]:
from openplaces.core.schema import Entity, Source

In [ ]:
source = Source(
    source_id='massgis',
    portal_url='https://www.mass.gov/forms/massgis-request-statewide-parcel-data',
)
print(source)

In [ ]:
# Create an entity without a version: version defaults to date
entity = Entity('parcel', source)
print(entity)
print(entity.to_path())
print(entity.to_prefix())

In [ ]:
# Create an entity with a named version
entity = Entity('parcel', source, '2018')
print(entity)
print(entity.to_path())
print(entity.to_prefix())

## DataSet & Theme

In [ ]:
from openplaces.core.schema import DataSet, Theme

In [ ]:
theme = Theme('land-terrain')
print(theme)
print(theme.to_path())
print(theme.to_prefix())

In [ ]:
theme = Theme('bio-species')
print(theme)
print(theme.to_path())
print(theme.to_prefix())

In [ ]:
dataset = DataSet(
    theme='land-terrain',
    source=Source(
        'srtm',
        'https://portal.opentopography.org/raster?opentopoID=OTSRTM.082015.4326.1',
    ),
    version='2000',
)
print(dataset)
print(dataset.to_path())
print(dataset.to_prefix())